# 01 · Pilot: decoding and alignment (design doc §7.4, revised; T4 version)

**How to run**
1. When choosing the kernel in VS Code, pick a Colab server with a GPU (a T4 is enough).
2. Click "Run All". The whole notebook takes about 2 hours.
3. Results go to `DATA_DIR/01_pilot_decoding/`. If the connection drops, click "Run All" again; finished parts are skipped.

**What it computes**: 100 MAGE human texts are cut to their first 30 tokens, and OPT-2.7B, Qwen2.5-3B and Qwen2.5-3B-Instruct continue each one to the same length. Each model writes once with each of two decoding settings:
- `pure`: temperature 1.0, top-p 1.0, sampling from the model's distribution as is
- `topp`: temperature 0.8, top-p 0.95 (a common truncated-sampling setting)

GPT-2 XL then scores the continuation part only, and the Fast-DetectGPT AUROCs are compared. MAGE's OPT-2.7B texts reach an AUROC of 0.235 under GPT-2 XL (a reversed direction); this experiment checks whether the decoding setting reproduces that number.

In [ ]:
# 1. Check for a GPU. The largest model here is 3B, so a T4 (15 GB) is enough.
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to a GPU"
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(name, f"{mem:.0f} GB")
assert mem > 13, f"This {name} has only {mem:.0f} GB; use a T4 or a larger GPU."

In [ ]:
# 2. Mount Google Drive and switch to the data folder; models are downloaded to the Colab machine, not to Drive.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.makedirs("01_pilot_decoding", exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"

In [ ]:
# 3. Install packages.
%pip install -q -U transformers huggingface-hub accelerate

In [ ]:
# 4. Pick human texts: 20 from each of 5 domains, 120 to 300 English words, fixed seed. Reuse the file if it exists.
import json
import pandas as pd
DOMAINS = ["xsum", "eli5", "yelp", "wp", "sci_gen"]
if not os.path.exists("01_pilot_decoding/humans.json"):
    !curl -sSL --retry 5 -o /content/test.csv https://huggingface.co/datasets/yaful/MAGE/resolve/main/test.csv
    d = pd.read_csv("/content/test.csv").dropna(subset=["text"]).drop_duplicates("text")
    d = d[d.src.isin([f"{x}_human" for x in DOMAINS])]
    words = d.text.str.split().str.len()
    d = d[(words >= 120) & (words <= 300)]
    pick = pd.concat([g.sample(20, random_state=0) for _, g in d.groupby("src")])
    rows = [dict(id=i, domain=s.replace("_human", ""), text=t) for i, (s, t) in enumerate(zip(pick.src, pick.text))]
    json.dump(rows, open("01_pilot_decoding/humans.json", "w"), ensure_ascii=False)
humans = json.load(open("01_pilot_decoding/humans.json"))
print(len(humans), "human texts")

In [ ]:
# 5. Continuations: 3 generators x 2 decoding settings, about 1.5 hours. Each finished condition is saved and skipped on a rerun.
import gc, shutil, traceback, time
import token_info

CONDITIONS = [  # (generator, decoding tag, temperature, top_p)
    ("facebook/opt-2.7b", "pure", 1.0, 1.0),
    ("facebook/opt-2.7b", "topp", 0.8, 0.95),
    ("Qwen/Qwen2.5-3B", "pure", 1.0, 1.0),
    ("Qwen/Qwen2.5-3B", "topp", 0.8, 0.95),
    ("Qwen/Qwen2.5-3B-Instruct", "pure", 1.0, 1.0),
    ("Qwen/Qwen2.5-3B-Instruct", "topp", 0.8, 0.95),
]
PREFIX = 30

for n, (model, dec, temp, top_p) in enumerate(CONDITIONS):
    path = f"01_pilot_decoding/gen_{model.split('/')[1]}_{dec}.json"
    if os.path.exists(path):
        print("Already done, skipping:", path)
        continue
    engine = None
    try:
        token_info.SETTINGS.update(
            scoring_model=model, sampling_model=model, temperature=temp, top_p=top_p, do_sample=True,
            prefix_tokens=PREFIX, paraphrase_enabled=False, save_token_features=False,
            cache_dir="/content/hf", seed=42)
        engine = token_info.TokenInfo()
        rows, start = [], time.time()
        for h in humans:
            try:
                r = engine.continue_text(h["text"])
            except (ValueError, RuntimeError) as e:
                print("Skipping text", h["id"], ":", e)
                continue
            ids = engine.encode(h["text"])
            rows.append(dict(id=h["id"], domain=h["domain"], model=model, decoding=dec, prefix=r["prefix"],
                             ai=r["continuation"],
                             human=engine.scoring_tokenizer.decode(ids[PREFIX:], clean_up_tokenization_spaces=False)))
            if len(rows) % 20 == 0:
                print(f"  {model} {dec}: {len(rows)}/{len(humans)} texts, {(time.time() - start) / 60:.0f} min so far", flush=True)
        json.dump(rows, open(path, "w"), ensure_ascii=False)
        print(model, dec, "done:", len(rows), "texts")
    except Exception:
        traceback.print_exc()
        print("This condition failed, skipping:", model, dec)
    finally:
        engine = None
        gc.collect(); torch.cuda.empty_cache()
    # After the last condition of a model, delete its files so the Colab disk does not fill up.
    if n + 1 == len(CONDITIONS) or CONDITIONS[n + 1][0] != model:
        shutil.rmtree(f"/content/hf/models--{model.replace('/', '--')}", ignore_errors=True)

In [ ]:
# 6. Score with GPT-2 XL; store surprisal, entropy, log-probability variance and top-10 mass at every position.
import glob
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

if not os.path.exists("01_pilot_decoding/scores.pkl"):
    tok = AutoTokenizer.from_pretrained("gpt2-xl")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
    scorer = AutoModelForCausalLM.from_pretrained("gpt2-xl", dtype=dtype).cuda().eval()
    recs = []
    for path in sorted(glob.glob("01_pilot_decoding/gen_*.json")):
        for r in json.load(open(path)):
            k = len(tok(r["prefix"])["input_ids"])   # prefix tokens, skipped when scoring
            for who in ("human", "ai"):
                ids = tok(r["prefix"] + r[who], truncation=True, max_length=512)["input_ids"]
                recs.append(dict(id=r["id"], domain=r["domain"], model=r["model"], decoding=r["decoding"],
                                 who=who, k=k, ids=ids))
    with torch.no_grad():
        for i, r in enumerate(recs):
            x = torch.tensor([r["ids"]]).cuda()
            lp = torch.log_softmax(scorer(x).logits[0, :-1].float(), -1)
            p = lp.exp()
            e1 = (p * lp).sum(-1)
            rows = [-lp.gather(1, x[0, 1:, None])[:, 0], -e1, (p * lp * lp).sum(-1) - e1 ** 2,
                    p.topk(10, -1).values.sum(-1)]
            r["arr"] = torch.stack(rows).cpu().numpy().astype(np.float32)[:, r["k"] - 1:]   # keep the continuation only
            if i % 200 == 0:
                print(i, "/", len(recs), flush=True)
    pd.DataFrame(recs).drop(columns="ids").to_pickle("01_pilot_decoding/scores.pkl")
    del scorer; gc.collect(); torch.cuda.empty_cache()
print("Scoring done")

In [ ]:
# 7. Quick summary: AUROC per condition (human vs AI continuations). The full analysis is scripts/a02_decoding_alignment.py.
from sklearn.metrics import roc_auc_score
s = pd.read_pickle("01_pilot_decoding/scores.pkl")
s["fdg"] = [(a[1].sum() - a[0].sum()) / np.sqrt(a[2].sum()) for a in s.arr]   # single-model Fast-DetectGPT; higher = more AI-like
s["y"] = (s.who == "ai").astype(int)
summary = s.groupby(["model", "decoding"]).apply(
    lambda g: pd.Series({"n_texts": len(g) // 2, "Fast-DetectGPT AUROC": roc_auc_score(g.y, g.fdg)}))
summary.to_csv("01_pilot_decoding/summary.csv")
print("Reference: MAGE OPT-2.7B texts score 0.235 under GPT-2 XL")
summary.round(3)

In [ ]:
# 8. Flush unsynced files back to Google Drive.
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/01_pilot_decoding/")